# YuE2 — save Kaggle assets once (revision 9)

**Internet ON; accelerator None.** No GPU model or song is run.
Downloads go directly to Kaggle, not through your home connection.
This saves the pinned YuE2/VAE, SheetSage2/MERT, optional Whisper,
Python overlay wheels, bootstrap pip and Gradio tunnel executable.
Weights are saved once without a duplicate Hub blobs tree.

Keep this notebook **private**. Use **Save Version → Save & Run All**
with outputs to build and preserve `/kaggle/working/yue2-assets` in one
saved run. Wait for **COMPLETE** in its logs. No test song follows.
You do not need to run it interactively first and repeat the download.
In your music notebook choose **Add Input → Notebook Output Files**
and select the completed saved builder version. That is already reusable
Kaggle storage; you do not need to download/re-upload any files.

If you specifically want a named **Dataset**, use the optional publish
cell below from this Kaggle session. It uploads directly from Kaggle to
your account and creates a **private** dataset. The normal generator
accepts either the dataset or saved notebook output automatically.

Keep the same Python minor version and preinstalled scientific package
versions in builder and generator. If Kaggle changes the image, the
generator explains the mismatch before any model work. Rebuilding on
the new image may be necessary. This is not a portable copy of all CUDA.
Enough output storage for all models is required; an interrupted build
has no complete assets.json and cannot be mistaken for a finished bundle.
Model licenses/cards are retained; upstream terms still apply.


In [ ]:
import base64, json, os, sys, zlib, subprocess
from pathlib import Path
support = Path("/kaggle/temp/yue2_asset_builder_support")
support.mkdir(parents=True, exist_ok=True)
payload = "..."
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    path = support / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
for name in list(sys.modules):
    if name.split(".")[0] in {"native_yue2", "kaggle_yue2"}:
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from native_yue2.setup import install
BUILDER_ROOT = "/kaggle/temp/yue2_asset_builder"
ASSET_OUTPUT = "/kaggle/working/yue2-assets"
native_python, native_env = install(
    BUILDER_ROOT, "/kaggle/temp/yue2_asset_builder_records/runs",
    assets=None, require_gpu=False)
subprocess.run([native_python, "-u", "-m", "native_yue2.assets",
                "--build", ASSET_OUTPUT, "--runtime-root", BUILDER_ROOT],
               env=native_env, check=True)


**Optional: create a named private dataset, directly from Kaggle.**
Set DATASET_HANDLE to `your-kaggle-username/yue2-music-assets` and run
the next cell. Leave it as None if using saved notebook outputs instead.
KaggleHub needs your Kaggle account authentication. If it requests a
token, configure Kaggle authentication privately; never paste credentials
into this notebook. New datasets are private by default in this API;
choose a new unused slug to avoid updating an older public dataset.
After upload, attach it using **Add Input → Your Datasets**. Keep the
dataset private. No model files travel through your home computer.


In [ ]:
DATASET_HANDLE = None  # Example: "janpierrrewiet/yue2-music-assets"
if DATASET_HANDLE:
    from native_yue2.assets import validate_assets
    validate_assets(ASSET_OUTPUT, full=True)
    import kagglehub
    if not hasattr(kagglehub, "dataset_upload"):
        raise RuntimeError("This Kaggle image needs a newer KaggleHub for dataset_upload. "
                           "Use saved notebook outputs now, or update KaggleHub in this builder only.")
    kagglehub.dataset_upload(
        DATASET_HANDLE, ASSET_OUTPUT,
        version_notes="Pinned YuE2 music + cover models and offline dependency wheels")
    print(f"Saved: https://www.kaggle.com/datasets/{DATASET_HANDLE}. Keep this dataset private.")
